# Feature Selection — Word2Vec

Word2Vec vektörleştirme üzerine 5 farklı özellik seçimi yönteminin etkisi.
Chi2, Mutual Information, LASSO, RFI, RFE × LR, DT, KNN, SVM, NC, GB, XGB, RF

### Kütüphaneler

In [1]:
import pandas as pd
import numpy as np
import time
import copy
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import xgboost as xgb
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif, RFE, SelectFromModel


### Veri Yükleme ve Bölme

In [2]:
df = pd.read_csv('spam_cleaned.csv', encoding='latin-1')

X, y = df['sms'], df['type']
print(f"Veri seti: {len(df)} satır | ham: {(y==0).sum()}, spam: {(y==1).sum()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=111, stratify=y
)
print(f"Eğitim: {len(X_train)} | Test: {len(X_test)} | Train spam oranı: {y_train.mean()*100:.1f}%")


Veri seti: 5570 satır | ham: 4823, spam: 747
Eğitim: 3899 | Test: 1671 | Train spam oranı: 13.4%


### Word2Vec Vektörleştirme

In [3]:
import gensim.downloader as api
print("Word2Vec modeli yükleniyor (word2vec-google-news-300)...")
w2v_model = api.load("word2vec-google-news-300")
print("Yüklendi.")

def sms_to_vec(text, model, dim=300):
    tokens = str(text).lower().split()
    vecs = [model[w] for w in tokens if w in model]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

X_train_vec = np.array([sms_to_vec(t, w2v_model) for t in X_train])
X_test_vec  = np.array([sms_to_vec(t, w2v_model) for t in X_test])
X_train_vec = np.abs(X_train_vec)
X_test_vec  = np.abs(X_test_vec)
print(f"Word2Vec özellik boyutu: {X_train_vec.shape[1]}")


Word2Vec modeli yükleniyor (word2vec-google-news-300)...
Yüklendi.
Word2Vec özellik boyutu: 300


### Feature Selection (5 Yöntem)

Chi2, Mutual Information, LASSO, Random Forest Importance, RFE — her biri k=100 özellik seçer.

In [4]:
k = min(100, X_train_vec.shape[1])

# Chi-Square
skb_chi2 = SelectKBest(score_func=chi2, k=k)
X_tr_chi2 = skb_chi2.fit_transform(np.abs(X_train_vec), y_train)
X_te_chi2 = skb_chi2.transform(np.abs(X_test_vec))

# Mutual Information
skb_mi = SelectKBest(score_func=mutual_info_classif, k=k)
X_tr_mi = skb_mi.fit_transform(X_train_vec, y_train)
X_te_mi = skb_mi.transform(X_test_vec)

# LASSO
lasso_clf = LogisticRegression(C=1.0, penalty='l1', solver='liblinear', max_iter=1000)
lasso_clf.fit(X_train_vec, y_train)
sel_lasso = SelectFromModel(lasso_clf, prefit=True)
X_tr_lasso = sel_lasso.transform(X_train_vec)
X_te_lasso = sel_lasso.transform(X_test_vec)

# Random Forest Importance
rf_sel = RandomForestClassifier(n_estimators=100, random_state=42)
rf_sel.fit(X_train_vec, y_train)
sel_rfi = SelectFromModel(rf_sel, prefit=True, max_features=100)
X_tr_rfi = sel_rfi.transform(X_train_vec)
X_te_rfi = sel_rfi.transform(X_test_vec)

# RFE
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=k, step=50)
X_tr_rfe = rfe.fit_transform(X_train_vec, y_train)
X_te_rfe = rfe.transform(X_test_vec)

print(f"Seçilen özellik sayıları — Chi2:{X_tr_chi2.shape[1]}, MI:{X_tr_mi.shape[1]}, "
      f"LASSO:{X_tr_lasso.shape[1]}, RFI:{X_tr_rfi.shape[1]}, RFE:{X_tr_rfe.shape[1]}")


Seçilen özellik sayıları — Chi2:100, MI:100, LASSO:57, RFI:58, RFE:100


### Değerlendirme Fonksiyonu

In [5]:
import scipy.stats as stats

def evaluate(name, clf, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred = clf.predict(X_te)
    try:
        y_score = clf.predict_proba(X_te)[:, 1]
    except AttributeError:
        y_score = clf.decision_function(X_te)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='accuracy')
    cv_acc = cv_scores.mean()
    cv_std = cv_scores.std()
    ci = stats.t.interval(0.95, df=len(cv_scores)-1, loc=cv_acc, scale=stats.sem(cv_scores))
    return {
        'Model':        name,
        'Accuracy':     round(accuracy_score(y_te, y_pred), 4),
        'Precision':    round(precision_score(y_te, y_pred, zero_division=0), 4),
        'Recall':       round(recall_score(y_te, y_pred), 4),
        'F1':           round(f1_score(y_te, y_pred), 4),
        'ROC-AUC':      round(roc_auc_score(y_te, y_score), 4),
        'CV-Acc':       round(cv_acc, 4),
        'CV-Std':       round(cv_std, 4),
        'CI-95% Low':   round(ci[0], 4),
        'CI-95% High':  round(ci[1], 4),
        'Time(s)':      round(elapsed, 4),
    }

### Tüm FS Yöntemleri × Tüm Sınıflandırıcılar

Her feature selection yöntemi ile 8 sınıflandırıcının karşılaştırmalı sonuçları.

In [7]:
fs_sets = {
    'Chi2':  (X_tr_chi2,  X_te_chi2),
    'MI':    (X_tr_mi,    X_te_mi),
    'LASSO': (X_tr_lasso, X_te_lasso),
    'RFI':   (X_tr_rfi,   X_te_rfi),
    'RFE':   (X_tr_rfe,   X_te_rfe),
}

classifiers = [
    ('LR',  LogisticRegression(max_iter=1000)),
    ('DT',  DecisionTreeClassifier(random_state=42)),
    ('KNN', KNeighborsClassifier(n_neighbors=3)),
    ('SVM', SVC(kernel='linear', probability=True)),
    ('NC',  NearestCentroid()),
    ('GB',  GradientBoostingClassifier(random_state=42)),
    ('XGB', xgb.XGBClassifier(eval_metric='logloss', random_state=42)),
    ('RF',  RandomForestClassifier(n_estimators=100, random_state=42)),
]

all_rows = []
total = len(fs_sets) * len(classifiers)
done  = 0

for fs_name, (X_tr, X_te) in fs_sets.items():
    for clf_name, clf in classifiers:
        t0 = time.time()
        row = evaluate(clf_name, copy.deepcopy(clf), X_tr, y_train, X_te, y_test)
        row['FeatureSelection'] = fs_name
        all_rows.append(row)
        done += 1
        elapsed = time.time() - t0
        print(f"[{done:2d}/{total}] {fs_name:6s} + {clf_name:3s}  "
              f"Acc={row['Accuracy']:.4f}  F1={row['F1']:.4f}  ({elapsed:.1f}s)")
    print()

cols = ['FeatureSelection', 'Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'CV-Acc', 'CV-Std', 'CI-95% Low', 'CI-95% High', 'Time(s)']

df_results = pd.DataFrame(all_rows)[cols]


[ 1/40] Chi2   + LR   Acc=0.9677  F1=0.8708  (0.0s)
[ 2/40] Chi2   + DT   Acc=0.9425  F1=0.7913  (1.5s)
[ 3/40] Chi2   + KNN  Acc=0.9629  F1=0.8686  (0.0s)
[ 4/40] Chi2   + SVM  Acc=0.9755  F1=0.9053  (1.0s)
[ 5/40] Chi2   + NC   Acc=0.9521  F1=0.8319  (0.0s)
[ 6/40] Chi2   + GB   Acc=0.9749  F1=0.9023  (34.1s)
[ 7/40] Chi2   + XGB  Acc=0.9785  F1=0.9167  (1.3s)
[ 8/40] Chi2   + RF   Acc=0.9713  F1=0.8824  (6.4s)

[ 9/40] MI     + LR   Acc=0.9677  F1=0.8714  (0.0s)
[10/40] MI     + DT   Acc=0.9414  F1=0.7851  (1.4s)
[11/40] MI     + KNN  Acc=0.9629  F1=0.8681  (0.0s)
[12/40] MI     + SVM  Acc=0.9731  F1=0.8966  (1.1s)
[13/40] MI     + NC   Acc=0.9515  F1=0.8316  (0.0s)
[14/40] MI     + GB   Acc=0.9749  F1=0.9032  (33.7s)
[15/40] MI     + XGB  Acc=0.9773  F1=0.9120  (1.3s)
[16/40] MI     + RF   Acc=0.9707  F1=0.8808  (6.2s)

[17/40] LASSO  + LR   Acc=0.9755  F1=0.9021  (0.0s)
[18/40] LASSO  + DT   Acc=0.9408  F1=0.7795  (0.7s)
[19/40] LASSO  + KNN  Acc=0.9683  F1=0.8870  (0.0s)
[20/40] 

In [8]:
print("\n=== ÖZET ===")
print(df_results.to_string(index=False))


=== ÖZET ===
FeatureSelection Model  Accuracy  Precision  Recall     F1  ROC-AUC  CV-Acc  CV-Std  CI-95% Low  CI-95% High  Time(s)
            Chi2    LR    0.9677     0.9381  0.8125 0.8708   0.9847  0.9626  0.0069      0.9530       0.9721   0.0065
            Chi2    DT    0.9425     0.7712  0.8125 0.7913   0.8876  0.9397  0.0095      0.9265       0.9529   0.2684
            Chi2   KNN    0.9629     0.8266  0.9152 0.8686   0.9674  0.9702  0.0082      0.9589       0.9816   0.0006
            Chi2   SVM    0.9755     0.9378  0.8750 0.9053   0.9873  0.9708  0.0058      0.9627       0.9788   0.2240
            Chi2    NC    0.9521     0.7857  0.8839 0.8319   0.9705  0.9379  0.0113      0.9223       0.9536   0.0015
            Chi2    GB    0.9749     0.9417  0.8661 0.9023   0.9842  0.9761  0.0060      0.9678       0.9845   6.8701
            Chi2   XGB    0.9785     0.9519  0.8839 0.9167   0.9832  0.9777  0.0050      0.9707       0.9847   0.2292
            Chi2    RF    0.9713     0.978